# 🧪 Thực Nghiệm Test 2 & Test 3: Kháng Sụp Đổ Softmax Entropy & Độ Ổn Định Lipschitz Vector Dẫn Đường
### **Khung Thực Nghiệm**: So Sánh LiDAR ($\sigma=0$) vs RS-LiDAR ($\sigma \in \{0.5, 1.0, 2.0\}, M=4$) trên Benchmark GenEval

Notebook này được tinh chỉnh tối giản **chuyên biệt cho Bài Test 2 & Test 3**, tập trung giải quyết các câu hỏi khoa học cốt lõi:
1. **Test 2 (Bóc trần hiện tượng Best-of-1 Trap của LiDAR)**: Khi $\lambda=5000$, hàm Softmax bão hòa cực đoan, dồn $>99\%$ trọng số vào đúng 1 hạt duy nhất khiến Shannon Entropy $H \to 0\text{ bits}$ (lãng phí công sức tính toán của toàn bộ các hạt còn lại).
2. **Test 3 (Độ ổn định Lipschitz của Vector dẫn đường)**: Khảo sát độ tương đồng Cosine $\text{CosSim}(\mathbf{g}_t, \mathbf{g}_{t+\delta})$ trước và sau khi thêm nhiễu vi mô $\delta$ ($\|\delta\|_2 = 10^{-3}$) dọc theo quỹ đạo khử nhiễu Phase 2. Chứng minh RS-LiDAR triệt tiêu rung giật gradient.
3. **Khảo sát Bán kính Nhiễu $\sigma \in \{0.5, 1.0, 2.0\}$**: Kiểm chứng định lý Dimension-Free Lipschitz Bound theo từng mức độ làm mịn.
4. **Minh chứng trực quan**: Tự động xác định và **lưu lại chính xác bức ảnh của hạt mà LiDAR bị sụp đổ về** để đối chứng trực tiếp.

## 1. Kiểm Tra Phần Cứng GPU & Cấu Hình Bộ Nhớ
Đảm bảo đã chọn **GPU T4 x2** (hoặc GPU T4 x1) trong cột cài đặt bên phải Kaggle (Session options -> Accelerator).

In [ ]:
import os, sys, torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA khả dụng: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"🎮 Số lượng GPU phát hiện: {n_gpus}")
    for i in range(n_gpus):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  • GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.2f} GB VRAM)")
else:
    raise RuntimeError("❌ Không phát hiện GPU CUDA! Vui lòng bật GPU trong Settings -> Accelerator -> GPU T4 x2.")

# Chống phân mảnh bộ nhớ VRAM trên GPU T4 16GB
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
!nvidia-smi

## 2. Đồng Bộ Mã Nguồn Repo & Cài Đặt Thư Viện Cần Thiết
Chỉ cài đặt đúng các thư viện thiết yếu cho Lookahead và Test 2 (Stable Diffusion, Diffusers, ImageReward).

In [ ]:
import os, shutil, glob, json, random, subprocess, threading

# 1. Đồng bộ repo
REPO_DIR = "/kaggle/working/RS-LiDAR"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/leekwanreal/RS-LiDAR.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main

WORKDIR = f"{REPO_DIR}/Diffusion-LiDAR-Sampling" if os.path.exists(f"{REPO_DIR}/Diffusion-LiDAR-Sampling") else REPO_DIR
os.chdir(WORKDIR)
%cd {WORKDIR}
print("📂 Thư mục làm việc:", os.getcwd())

# 2. Cài đặt các thư viện cần thiết
!pip install -q --upgrade protobuf
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q scipy matplotlib seaborn pandas tabulate

# 3. Hàm tiện ích chạy song song 2 GPU hiển thị log trực tiếp
def run_commands_parallel(cmd0, cmd1):
    p0 = subprocess.Popen(cmd0, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    p1 = subprocess.Popen(cmd1, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    def stream_logs(proc, prefix):
        for line in iter(proc.stdout.readline, ''):
            if line.strip():
                print(f"{prefix} {line.strip()}")
        proc.stdout.close()
    t0 = threading.Thread(target=stream_logs, args=(p0, "[GPU 0]"))
    t1 = threading.Thread(target=stream_logs, args=(p1, "[GPU 1]"))
    t0.start(); t1.start()
    t0.join(); t1.join()
    rc0 = p0.wait()
    rc1 = p1.wait()
    if rc0 == 0:
        print("✅ [GPU 0] Tiến trình hoàn thành thành công.")
    else:
        print(f"❌ [GPU 0] Tiến trình dừng với mã lỗi exit code = {rc0}!")
    if rc1 == 0:
        print("✅ [GPU 1] Tiến trình hoàn thành thành công.")
    else:
        print(f"❌ [GPU 1] Tiến trình dừng với mã lỗi exit code = {rc1}!")

print("\n✅ Môi trường thực nghiệm Test 2 đã sẵn sàng 100%!")

## 3. Cấu Hình Tham Số Thực Nghiệm & Lấy Mẫu Prompts GenEval
**Khu vực cấu hình tập trung**: Bạn chỉ cần điều chỉnh các biến tham số ở cell bên dưới (`NUM_PROMPTS`, `NUM_PARTICLES`, `SIGMA`, `SIGMAS_SWEEP`, `NUM_MC_SAMPLES`), toàn bộ các cell sau sẽ tự động sử dụng cấu hình mới.

In [ ]:
# ==============================================================================
# ⚙️ KHU VỰC CẤU HÌNH THAM SỐ THỰC NGHIỆM (CHỈ CẦN CHỈNH SỬA TẠI ĐÂY)
# ==============================================================================
NUM_PROMPTS = 20                  # Số lượng prompt thực nghiệm (20 prompts theo yêu cầu)
NUM_PARTICLES = 50                # Số lượng hạt lookahead mỗi prompt (N = 50)
LOOKAHEAD_STEPS = 5               # Số bước DPM-Solver lookahead ở Phase 1 (5 bước)
LOOKAHEAD_SEED = 100              # Seed sinh hạt lookahead (DPM-5 seed 100)
PROMPT_SEED = 42                  # Seed chọn prompt ngẫu nhiên từ GenEval (đảm bảo 100% tái lập)

SIGMA = 1.0                       # Độ lệch chuẩn chính cho RS-LiDAR (σ = 1.0)
SIGMAS_SWEEP = "0.5,1.0,2.0"       # Dải khảo sát độ lệch chuẩn Randomized Smoothing (σ ∈ {0.5, 1.0, 2.0})
NUM_MC_SAMPLES = 4                # M = 4 mẫu Monte Carlo để tính kỳ vọng E[r(x+xi)]
# ==============================================================================

PROMPT_FILE = "prompt_files/geneval_metadata.jsonl"
TEST_PROMPT_FILE = f"/kaggle/working/test_{NUM_PROMPTS}_prompts.jsonl"
TEST2_PROMPT_FILE = TEST_PROMPT_FILE  # Tương thích ngược

# Đọc toàn bộ prompts GenEval
with open(PROMPT_FILE, "r", encoding="utf-8") as f:
    all_prompts = [json.loads(line) for line in f if line.strip()]

print(f"📚 Tổng số prompts trong GenEval: {len(all_prompts)}")

# Chọn NUM_PROMPTS ngẫu nhiên (cố định seed để tái lập)
random.seed(PROMPT_SEED)
selected_prompts = random.sample(all_prompts, min(NUM_PROMPTS, len(all_prompts)))

with open(TEST_PROMPT_FILE, "w", encoding="utf-8") as f:
    for item in selected_prompts:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"✅ Đã chọn và lưu {len(selected_prompts)} prompts ngẫu nhiên vào: {TEST_PROMPT_FILE}\n")
print(f"{'STT':<5} {'Tag / Chủ đề':<18} {'Nội dung Prompt'}")
print("-" * 85)
for i, p in enumerate(selected_prompts):
    tag = p.get('tag', 'general')
    text = p.get('prompt', '')
    print(f"{i+1:<5} {tag:<18} {text}")

## 4. [PHASE 1] Sinh Hạt Lookahead Thật (DPM-Solver 5 Bước)
Mỗi prompt sinh các hạt Lookahead thật bằng DPM-Solver 5 bước và bật cờ `--save_individual_images True` để lưu toàn bộ ảnh hạt mẫu vào ổ đĩa.

* *Thời gian ước tính*: ~1-2 phút cho 10 prompts khi chạy song song 2x GPU T4.

In [ ]:
LOOKAHEAD_TAG = f"{LOOKAHEAD_SEED}_{NUM_PARTICLES}_{LOOKAHEAD_STEPS}"
LOOKAHEAD_DIR = f"/kaggle/working/Lookahead_samples/{LOOKAHEAD_TAG}"
os.makedirs(LOOKAHEAD_DIR, exist_ok=True)

# Kiểm tra xem các prompt đã được tạo từ trước hay chưa
n_done = len(glob.glob(f"{LOOKAHEAD_DIR}/[0-9]*/results.json"))
OVERWRITE_PHASE1 = False  # Đổi thành True nếu bạn muốn xóa chạy lại Phase 1 từ đầu

if n_done >= NUM_PROMPTS and not OVERWRITE_PHASE1:
    print(f"⏩ [SKIP PHASE 1] Đã hoàn thành đủ {n_done}/{NUM_PROMPTS} prompt Lookahead từ trước!")
    print(f"👉 Chuyển thẳng sang Cell tiếp theo (Phase 2 - Test 2) để đo Entropy và xem ảnh sụp đổ.")
else:
    # Dọn dẹp tiến trình và bộ nhớ đệm cũ
    import gc
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    !pkill -f "lookahead_sampling.py" || true

    # 0. Tiền tải weights chuẩn Diffusers vào cache ổ đĩa (Rank 0)
    print("📥 Đang kiểm tra / tiền tải model weights chuẩn Diffusers vào cache (chỉ ~3.9 GB, không tải file 7.7 GB của WebUI)...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        "runwayml/stable-diffusion-v1-5",
        allow_patterns=[
            "model_index.json",
            "*scheduler/*",
            "*tokenizer*/*",
            "*text_encoder*/*.json",
            "*text_encoder*/*model.safetensors",
            "*unet*/*.json",
            "*unet*/*diffusion_pytorch_model.safetensors",
            "*vae*/*.json",
            "*vae*/*diffusion_pytorch_model.safetensors",
            "*feature_extractor/*"
        ],
        ignore_patterns=["*.ckpt", "*pruned*", "*non_ema*", "safety_checker/*", "*example-lora*", "*.bin", "*fp16*"]
    )

    ir_weight_path = os.path.expanduser("~/.cache/ImageReward/ImageReward.pt")
    if not os.path.exists(ir_weight_path):
        os.makedirs(os.path.dirname(ir_weight_path), exist_ok=True)
        import urllib.request
        try:
            urllib.request.urlretrieve("https://huggingface.co/THUDM/ImageReward/resolve/main/ImageReward.pt", ir_weight_path)
            urllib.request.urlretrieve("https://huggingface.co/THUDM/ImageReward/resolve/main/med_config.json", os.path.join(os.path.dirname(ir_weight_path), "med_config.json"))
        except Exception as e:
            print(f"Lưu ý: Không thể tải trước ImageReward ({e}), ImageReward sẽ tự động tải khi chạy.")
    print("✅ Weights đã có sẵn trên đĩa! Bắt đầu thực thi song song 2 GPU hoàn toàn không bị nghẽn mạng hay đợi model.")

    n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
    ov_arg = "--overwrite" if OVERWRITE_PHASE1 else ""

    if n_gpus >= 2:
        print(f"🚀 [2 GPU] Đang chạy Lookahead Sampling song song trên GPU 0 và GPU 1 ({NUM_PROMPTS} prompts tổng cộng)...")
        cmd0 = f"""python -u lookahead_sampling.py \
            --prompt_path '{TEST2_PROMPT_FILE}' \
            --model_name 'runwayml/stable-diffusion-v1-5' \
            --num_particles {NUM_PARTICLES} \
            --num_inference_steps {LOOKAHEAD_STEPS} \
            --seed {LOOKAHEAD_SEED} \
            --save_individual_images True \
            --output_dir '/kaggle/working/Lookahead_samples' \
            --num_shards 2 --shard_id 0 --gpu_id 0 {ov_arg}"""
            
        cmd1 = f"""python -u lookahead_sampling.py \
            --prompt_path '{TEST2_PROMPT_FILE}' \
            --model_name 'runwayml/stable-diffusion-v1-5' \
            --num_particles {NUM_PARTICLES} \
            --num_inference_steps {LOOKAHEAD_STEPS} \
            --seed {LOOKAHEAD_SEED} \
            --save_individual_images True \
            --output_dir '/kaggle/working/Lookahead_samples' \
            --num_shards 2 --shard_id 1 --gpu_id 1 {ov_arg}"""
        run_commands_parallel(cmd0, cmd1)
    else:
        print(f"🚀 [1 GPU] Đang chạy Lookahead Sampling trên 1 GPU ({NUM_PROMPTS} prompts)...")
        !python -u lookahead_sampling.py \
            --prompt_path '{TEST2_PROMPT_FILE}' \
            --model_name 'runwayml/stable-diffusion-v1-5' \
            --num_particles {NUM_PARTICLES} \
            --num_inference_steps {LOOKAHEAD_STEPS} \
            --seed {LOOKAHEAD_SEED} \
            --save_individual_images True \
            --output_dir '/kaggle/working/Lookahead_samples' {ov_arg}

    n_done_now = len(glob.glob(f"{LOOKAHEAD_DIR}/*/results.json"))
    print(f"\n✅ HOÀN TẤT PHASE 1: Đã có đủ {n_done_now}/{NUM_PROMPTS} prompts tại {LOOKAHEAD_DIR}!")


## 5. [PHASE 2] Chạy Test 2 (Entropy Softmax) & Test 3 (Độ Ổn Định Vector Dẫn Đường)
Script nạp trực tiếp các hạt Lookahead và điểm ImageReward vừa sinh ở Phase 1, đo đường cong Entropy và độ ổn định Lipschitz dọc theo quỹ đạo khử nhiễu 50 bước với dải làm mịn RS-LiDAR: `sigmas = 0.5, 1.0, 2.0` và `M = NUM_MC_SAMPLES` mẫu Monte Carlo, đồng thời **trích xuất chính xác bức ảnh mà LiDAR bị sụp đổ về** vào thư mục `collapsed_images/`.

In [ ]:
TEST_OUTPUT_DIR = "/kaggle/working/test2_3_results"
TEST2_OUTPUT_DIR = TEST_OUTPUT_DIR  # Tương thích ngược
os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)

print(f"🔬 Đang khởi chạy Bài Test 2 & Test 3 với sigmas = {SIGMAS_SWEEP}, M = {NUM_MC_SAMPLES} trên {NUM_PROMPTS} prompts...")
!python test_lidar_weaknesses.py \
    --test 2,3 \
    --num_prompts {NUM_PROMPTS} \
    --num_particles {NUM_PARTICLES} \
    --sigma {SIGMA} \
    --tune_sigma \
    --sigmas "{SIGMAS_SWEEP}" \
    --M {NUM_MC_SAMPLES} \
    --prompt_path '{TEST_PROMPT_FILE}' \
    --lookahead_dir '{LOOKAHEAD_DIR}' \
    --output_dir '{TEST_OUTPUT_DIR}'

print(f"\n✅ HOÀN TẤT TEST 2 & TEST 3: Kết quả đã lưu tại {TEST_OUTPUT_DIR}!")

## 6. Hiển Thị Bảng Kết Quả Chi Tiết, Bảng Khảo Sát Sigma, Biểu Đồ & Trưng Bày Ảnh Sụp Đổ
Quan sát trực quan toàn bộ đầu ra khoa học: Bảng tổng hợp so sánh điểm yếu, Bảng khảo sát bán kính nhiễu $\sigma \in \{0.5, 1.0, 2.0\}$, Bảng chi tiết từng bước cho Test 2 & Test 3, đồ thị so sánh và ảnh của các hạt mà LiDAR bị sụp đổ về.

In [ ]:
import pandas as pd, glob, os
from IPython.display import display, Image, Markdown

# 1. Bảng Tổng Hợp So Sánh Khoa Học (Weaknesses Comparison Table)
comp_csvs = glob.glob(f"{TEST_OUTPUT_DIR}/weaknesses_comparison_table*.csv")
if comp_csvs:
    df_comp = pd.read_csv(comp_csvs[0])
    print("=" * 110)
    print("📊 1. BẢNG TỔNG HỢP SO SÁNH ĐIỂM YẾU LIDAR VS RS-LIDAR:")
    print("=" * 110)
    display(df_comp)

# 2. Bảng Khảo Sát Bán Kính Làm Mịn Sigma Ablation (σ ∈ {0.5, 1.0, 2.0})
abl_csvs = glob.glob(f"{TEST_OUTPUT_DIR}/sigma_ablation_table*.csv")
if abl_csvs:
    df_abl = pd.read_csv(abl_csvs[0])
    print("\n" + "=" * 110)
    print(f"📊 2. BẢNG KHẢO SÁT BÁN KÍNH NHIỄU SIGMA (σ ∈ {{{SIGMAS_SWEEP}}}):")
    print("=" * 110)
    display(df_abl)

# 3. Test 2: Bảng hạt sụp đổ theo từng prompt
t2_p_csvs = glob.glob(f"{TEST_OUTPUT_DIR}/test_2_prompt_collapsed_particles*.csv")
if t2_p_csvs:
    df_t2_p = pd.read_csv(t2_p_csvs[0])
    print("\n" + "=" * 110)
    print(f"📊 3. TEST 2 - BẢNG TỔNG HỢP HẠT SỤP ĐỔ CỦA LIDAR (BEST-OF-1 TRAP) TRÊN {len(df_t2_p)} PROMPTS:")
    print("=" * 110)
    display(df_t2_p)

# 4. Test 2: Bảng chi tiết sụp đổ entropy từng bước (50 steps)
t2_s_csvs = glob.glob(f"{TEST_OUTPUT_DIR}/test_2_entropy_collapse_by_step*.csv")
if t2_s_csvs:
    df_t2_s = pd.read_csv(t2_s_csvs[0])
    print("\n" + "=" * 110)
    print("📊 4. TEST 2 - CHI TIẾT SỰ SỤP ĐỔ ENTROPY THEO TỪNG BƯỚC KHỬ NHIỄU (50 STEPS):")
    print("=" * 110)
    display(df_t2_s.head(10))

# 5. Test 3: Bảng độ ổn định Lipschitz theo bước (50 steps)
t3_s_csvs = glob.glob(f"{TEST_OUTPUT_DIR}/test_3_cossim_stability_by_step*.csv")
if t3_s_csvs:
    df_t3_s = pd.read_csv(t3_s_csvs[0])
    print("\n" + "=" * 110)
    print("📊 5. TEST 3 - CHI TIẾT ĐỘ ỔN ĐỊNH LIPSCHITZ VECTOR DẪN ĐƯỜNG THEO BƯỚC (50 STEPS):")
    print("=" * 110)
    display(df_t3_s.head(10))

# 6. Test 3: Bảng tổng hợp độ ổn định theo từng prompt
t3_p_csvs = glob.glob(f"{TEST_OUTPUT_DIR}/test_3_prompt_stability_summary*.csv")
if t3_p_csvs:
    df_t3_p = pd.read_csv(t3_p_csvs[0])
    print("\n" + "=" * 110)
    print(f"📊 6. TEST 3 - TỔNG HỢP ĐỘ ỔN ĐỊNH COSIM TRÊN {len(df_t3_p)} PROMPTS:")
    print("=" * 110)
    display(df_t3_p.head(10))

# 7. Hiển thị tất cả Đồ thị biểu diễn
plot_files = sorted(glob.glob(f"{TEST_OUTPUT_DIR}/*.png"))
for pf in plot_files:
    print(f"\n📈 Đồ thị: {os.path.basename(pf)}")
    display(Image(filename=pf))

# 8. Trưng bày ảnh hạt sụp đổ của Test 2
collapsed_imgs = sorted(glob.glob(f"{TEST_OUTPUT_DIR}/collapsed_images/*lidar_collapse*.png"))
print(f"\n🖼️ TỔNG SỐ ẢNH HẠT SỤP ĐỔ ĐÃ LƯU: {len(collapsed_imgs)} ảnh trong thư mục 'collapsed_images/'")
print("Trưng bày ảnh sụp đổ tiêu biểu:")
for c_img in collapsed_imgs[:min(6, len(collapsed_imgs))]:
    print(f"  • File: {os.path.basename(c_img)}")
    display(Image(filename=c_img, width=256))

## 7. Đóng Gói Toàn Bộ Kết Quả & Ảnh Sụp Đổ (1-Click Download)
Nén toàn bộ bảng CSV, Markdown, đồ thị và toàn bộ ảnh hạt sụp đổ thành file zip `test2_3_results.zip` để tải về trực tiếp từ tab **Output** của Kaggle.

In [ ]:
zip_path = "/kaggle/working/test2_3_results.zip"
print(f"📦 Đang nén toàn bộ kết quả Test 2 & Test 3 vào {zip_path}...")

!zip -r -q {zip_path} {TEST_OUTPUT_DIR}

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"🎉 ĐÓNG GÓI THÀNH CÔNG! Dung lượng file: {size_mb:.2f} MB")
    print(f"📁 Đường dẫn file zip: {zip_path}")
    print("👉 Bạn có thể tải file này trực tiếp từ cột 'Output' bên phải giao diện Kaggle!")
else:
    print("⚠️ Không tìm thấy file zip được tạo.")